**Import Libraries**

In [15]:
import torch
import librosa
import numpy as np
import pandas as pd
import os
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import warnings

warnings.filterwarnings("ignore")

**Load the Wav2Vec2 Model**

In [16]:
print("Loading Wav2Vec2 Base model from Hugging Face...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
model.to(device)
model.eval()
print("Model loaded successfully!")

Loading Wav2Vec2 Base model from Hugging Face...
Using device: cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


**Extraction Function**

In [17]:
def extract_wav2vec_embedding(file_path):
    """Loads audio, resamples to 16kHz, and extracts a 2304-dim embedding."""
    
    try:
        # y  → the audio signal (the waveform), sr → the sampling rate
        y, sr = librosa.load(file_path, sr=16000)
        inputs = processor(y, sampling_rate=sr, return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)

        hidden = outputs.last_hidden_state  # shape: (1, time_steps, 768)
        
        mean_pool = hidden.mean(dim=1)
        max_pool = hidden.max(dim=1).values
        std_pool = hidden.std(dim=1)

        embeddings = torch.cat([mean_pool, max_pool, std_pool], dim=1).squeeze().detach().cpu().numpy()
        
        return embeddings
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

**Process the Audio Dataset**

In [18]:
df = pd.read_csv("../data/processed/cleaned_transcripts.csv")

features_list = []
valid_labels = []
valid_texts = []

print("Extracting 2304-dim Wav2Vec2 embeddings (this will take some time)...")

for index, row in df.iterrows():
    file_path = f"../data/processed/audio/sample_{index}.wav"
    
    if os.path.exists(file_path):
        embedding = extract_wav2vec_embedding(file_path)
        
        if embedding is not None:
            features_list.append(embedding)
            valid_labels.append(row['label'])
            valid_texts.append(row['text'])
            
    if (index + 1) % 50 == 0:
        print(f"Processed {index + 1} / {len(df)} files...")

print(f"Successfully extracted embeddings from {len(features_list)} files.")

Extracting 2304-dim Wav2Vec2 embeddings (this will take some time)...
Processed 50 / 800 files...
Processed 100 / 800 files...
Processed 150 / 800 files...
Processed 200 / 800 files...
Processed 250 / 800 files...
Processed 300 / 800 files...
Processed 350 / 800 files...
Processed 400 / 800 files...
Processed 450 / 800 files...
Processed 500 / 800 files...
Processed 550 / 800 files...
Processed 600 / 800 files...
Processed 650 / 800 files...
Processed 700 / 800 files...
Processed 750 / 800 files...
Processed 800 / 800 files...
Successfully extracted embeddings from 800 files.


**Save the Features to CSV**

In [19]:
embedding_dim = len(features_list[0])

col_names = [f'w2v_dim_{i}' for i in range(1, embedding_dim + 1)]
features_df = pd.DataFrame(features_list, columns=col_names)

features_df['text'] = valid_texts
features_df['label'] = valid_labels

final_path = "../data/processed/wav2vec_features.csv"
features_df.to_csv(final_path, index=False)

print(f"Saved production-ready audio dataset to: {final_path}")
features_df.head()

Saved production-ready audio dataset to: ../data/processed/wav2vec_features.csv


,w2v_dim_1,w2v_dim_2,w2v_dim_3,w2v_dim_4,w2v_dim_5,w2v_dim_6,w2v_dim_7,w2v_dim_8,w2v_dim_9,w2v_dim_10,...,w2v_dim_2297,w2v_dim_2298,w2v_dim_2299,w2v_dim_2300,w2v_dim_2301,w2v_dim_2302,w2v_dim_2303,w2v_dim_2304,text,label
0,-0.071143,0.118299,0.343188,0.064276,0.268205,-0.207779,0.034772,-0.198786,-0.193608,-0.152646,...,0.103407,0.400611,0.134892,0.057627,0.056545,0.135854,0.220571,0.149470,"[Greetings], this is [Name]. I finally got my ...",0
1,-0.049307,0.085180,0.266006,0.051329,0.326364,-0.217478,0.050141,-0.204784,-0.214295,-0.174420,...,0.109490,0.391991,0.133367,0.066850,0.052868,0.130854,0.227899,0.153363,"[Greetings], this is [Name]. I am hosting a sm...",0
2,-0.090469,0.069514,0.293868,0.078358,0.280848,-0.204684,0.033359,-0.217258,-0.234654,-0.169273,...,0.117451,0.428465,0.133934,0.059866,0.059002,0.137917,0.217539,0.150056,"[Greetings], this is the [Company] contacting...",1
3,-0.084973,0.088180,0.301088,0.081678,0.290748,-0.241272,0.053737,-0.183645,-0.201844,-0.174119,...,0.104686,0.391093,0.131440,0.061215,0.053382,0.124056,0.228507,0.153677,"[Greetings], this is [Name] from [Company] Ene...",0
4,-0.093775,0.065640,0.311353,0.087361,0.255466,-0.192279,0.067952,-0.175145,-0.238327,-0.171779,...,0.120890,0.442194,0.131457,0.063214,0.058520,0.146697,0.220018,0.158885,"[Greetings], this is the [Company] reaching o...",1
